# 01 · Data Cleaning & Feature Engineering
**Project:** Banking Risk Analytics & Customer Financial Behaviour

**Goal of this notebook:** turn the raw banking extract (3,000 client records × 25 columns + 3 lookup tables) into one clean, analysis-ready table that feeds the SQL analysis (PostgreSQL) and the Tableau dashboard.

**Steps**
1. Load raw data and lookup tables
2. Data-quality checks (types, missing values, duplicates, key integrity, zero balances, outliers)
3. Fix issues (parse dates, create a reliable surrogate key)
4. Join lookup tables (gender, banking relationship, investment advisor)
5. Feature engineering (total loan / deposit, income band, age group, tenure, processing fee, risk ratios, segments)
6. Export `banking_clean.csv`

In [1]:
import pandas as pd
import numpy as np
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.2f}".format)

RAW = "../data/raw/"
df = pd.read_csv(RAW + "Banking.csv")
gender = pd.read_csv(RAW + "gender.csv", encoding="utf-8-sig")
rel = pd.read_csv(RAW + "banking_relationships.csv", encoding="utf-8-sig")
adv = pd.read_csv(RAW + "investment_advisors.csv", encoding="utf-8-sig")
print("Clients table:", df.shape)
df.head()

Clients table: (3000, 25)


,Client ID,Name,Age,Location ID,Joined Bank,Banking Contact,Nationality,Occupation,Fee Structure,Loyalty Classification,Estimated Income,Superannuation Savings,Amount of Credit Cards,Credit Card Balance,Bank Loans,Bank Deposits,Checking Accounts,Saving Accounts,Foreign Currency Account,Business Lending,Properties Owned,Risk Weighting,BRId,GenderId,IAId
0,IND81288,Raymond Mills,24,34324,06-05-2019,Anthony Torres,American,Safety Technician IV,High,Jade,"75,384.77","17,677.95",1,484.54,"776,242.92","1,485,828.64","603,617.88","607,332.46","12,249.96","1,134,475.30",1,2,1,1,1
1,IND65833,Julia Spencer,23,42205,10-12-2001,Jonathan Hawkins,African,Software Consultant,High,Jade,"289,834.31","17,398.92",1,"2,256.88","1,270,615.43","641,482.79","229,521.37","344,635.16","61,162.31","2,000,526.10",1,3,2,1,2
2,IND47499,Stephen Murray,27,7314,25-01-2010,Anthony Berry,European,Help Desk Operator,High,Gold,"169,935.23","42,825.90",2,"4,568.74","1,052,715.84","1,033,401.59","652,674.69","203,054.35","79,071.78","548,137.58",1,3,3,2,3
3,IND72498,Virginia Garza,40,34594,28-03-2019,Steve Diaz,American,Geologist II,Mid,Silver,"356,808.11","5,473.15",2,"4,205.00","121,195.06","1,048,157.49","1,048,157.49","234,685.02","57,513.65","1,148,402.29",0,4,4,1,4
4,IND60181,Melissa Sanders,46,41269,20-07-2012,Shawn Long,American,Assistant Professor,Mid,Platinum,"130,711.68","48,077.60",1,"3,779.49","1,048,301.95","487,782.53","446,644.25","128,351.45","30,012.14","1,674,412.12",0,3,1,2,5


## 1. Structure and data types

In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 25 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Client ID                 3000 non-null   str    
 1   Name                      3000 non-null   str    
 2   Age                       3000 non-null   int64  
 3   Location ID               3000 non-null   int64  
 4   Joined Bank               3000 non-null   str    
 5   Banking Contact           3000 non-null   str    
 6   Nationality               3000 non-null   str    
 7   Occupation                3000 non-null   str    
 8   Fee Structure             3000 non-null   str    
 9   Loyalty Classification    3000 non-null   str    
 10  Estimated Income          3000 non-null   float64
 11  Superannuation Savings    3000 non-null   float64
 12  Amount of Credit Cards    3000 non-null   int64  
 13  Credit Card Balance       3000 non-null   float64
 14  Bank Loans         

`Joined Bank` is stored as text (format `DD-MM-YYYY`). Everything else has a sensible type. `BRId`, `GenderId` and `IAId` are foreign keys to the lookup tables.

In [3]:
for name, t in [("gender", gender), ("banking relationships", rel), ("investment advisors", adv)]:
    print(f"{name}: {t.shape}"); display(t.head())

gender: (2, 2)


,GenderId,Gender
0,1,Male
1,2,Female


banking relationships: (4, 2)


,BRId,Banking Relationship
0,1,Retail
1,2,Institutional
2,3,Private Bank
3,4,Commercial


investment advisors: (22, 2)


,IAId,Investment Advisor
0,1,Victor Dean
1,2,Jeremy Porter
2,3,Ernest Knight
3,4,Eric Shaw
4,5,Kevin Kim


## 2. Data-quality checks

In [4]:
print("Missing values in total:", df.isna().sum().sum())
print("Fully duplicated rows:  ", df.duplicated().sum())
print("Rows:                   ", len(df))
print("Unique Client IDs:      ", df["Client ID"].nunique())

Missing values in total: 0
Fully duplicated rows:   0
Rows:                    3000
Unique Client IDs:       2940


**Finding: `Client ID` is not unique.** There are 3,000 rows but only 2,940 distinct IDs. Let's see whether these are true duplicates (same person twice) or different people sharing an ID.

In [5]:
dup = df[df["Client ID"].duplicated(keep=False)].sort_values("Client ID")
print("Rows involved:", len(dup), "| IDs reused:", dup["Client ID"].nunique())
same_person = dup.groupby("Client ID")["Name"].nunique().eq(1).sum()
print("IDs where the rows are the same person:", same_person)
dup[["Client ID", "Name", "Age", "Joined Bank", "Nationality", "Estimated Income"]].head(6)

Rows involved: 119 | IDs reused: 59
IDs where the rows are the same person: 0


,Client ID,Name,Age,Joined Bank,Nationality,Estimated Income
1545,IND15088,Margaret Coleman,42,25-12-2010,American,"286,607.89"
740,IND15088,Betty Davis,56,14-03-2013,European,"67,454.44"
378,IND15955,Susan Dunn,20,08-12-2011,Asian,"175,521.36"
1081,IND15955,Victor Williams,21,29-05-2015,African,"113,678.37"
995,IND17019,Keith Fuller,70,25-10-2014,European,"210,198.07"
259,IND17019,Charles Richards,28,05-03-2015,Asian,"241,251.12"


The rows sharing an ID are **different people** (different names, ages, join dates). So they must **not** be de-duplicated: deleting them would lose 59 real clients.

**Business impact:** a "Total Clients = DISTINCTCOUNT(Client ID)" KPI (as used in many tutorials of this dataset) reports **2,940** clients and silently undercounts by 60. **Fix:** create a surrogate key `client_key` (1…3,000) and count rows / keys instead. In a real bank, this would be raised with the data-engineering team as a source-system key-integrity issue.

In [6]:
df.insert(0, "client_key", np.arange(1, len(df) + 1))
df["client_id_reused"] = df["Client ID"].duplicated(keep=False)
df["client_id_reused"].sum()

np.int64(119)

### Foreign-key integrity

In [7]:
print("GenderId values not in lookup:", set(df.GenderId) - set(gender.GenderId))
print("BRId values not in lookup:    ", set(df.BRId) - set(rel.BRId))
print("IAId values not in lookup:    ", set(df.IAId) - set(adv.IAId))

GenderId values not in lookup: set()
BRId values not in lookup:     set()
IAId values not in lookup:     set()


All keys match, so the joins will not create nulls.

### Dates

In [8]:
df["Joined Bank"] = pd.to_datetime(df["Joined Bank"], format="%d-%m-%Y")
df["Joined Bank"].agg(["min", "max"])

min   1995-01-03
max   2021-12-31
Name: Joined Bank, dtype: datetime64[us]

Clients joined between Jan 1995 and Dec 2021. To keep the analysis **reproducible**, tenure is measured up to a fixed snapshot date of **31-Dec-2021** (the last date in the data), not `TODAY()` (which would change every time the file is opened).

### Zero balances and value ranges

In [9]:
money_cols = ["Estimated Income", "Superannuation Savings", "Credit Card Balance", "Bank Loans",
              "Bank Deposits", "Checking Accounts", "Saving Accounts", "Foreign Currency Account", "Business Lending"]
pd.DataFrame({"zeros": (df[money_cols] == 0).sum(), "negatives": (df[money_cols] < 0).sum(),
              "min": df[money_cols].min(), "median": df[money_cols].median(), "max": df[money_cols].max()})

,zeros,negatives,min,median,max
Estimated Income,0,0,"15,919.48","142,313.48","522,330.26"
Superannuation Savings,0,0,"1,482.03","22,357.36","75,963.90"
Credit Card Balance,0,0,1.17,"2,560.81","13,991.99"
Bank Loans,27,0,0.00,"479,793.40","2,667,556.66"
Bank Deposits,34,0,0.00,"463,316.46","3,890,598.08"
Checking Accounts,34,0,0.00,"242,815.65","1,969,923.08"
Saving Accounts,34,0,0.00,"164,086.55","1,724,118.36"
Foreign Currency Account,0,0,45.00,"24,341.19","124,704.87"
Business Lending,14,0,0.00,"711,314.66","3,825,961.94"


- No negative balances.
- A few clients have **zero** loans, deposits or business lending. These are valid (a client may hold no product of that type), so they are kept.
- Balances are right-skewed (max ≫ median), which is normal for financial data. Outliers are **kept** (large clients are real and matter most for risk) but medians are reported alongside means.

In [10]:
q1, q3 = df[money_cols].quantile(0.25), df[money_cols].quantile(0.75)
iqr_outliers = ((df[money_cols] > q3 + 1.5 * (q3 - q1))).sum()
iqr_outliers.rename("rows above Q3 + 1.5·IQR")

Estimated Income             26
Superannuation Savings       19
Credit Card Balance          85
Bank Loans                   85
Bank Deposits               149
Checking Accounts           138
Saving Accounts             155
Foreign Currency Account     79
Business Lending             93
Name: rows above Q3 + 1.5·IQR, dtype: int64

## 3. Join the lookup tables

In [11]:
df = (df.merge(gender, on="GenderId", how="left")
        .merge(rel, on="BRId", how="left")
        .merge(adv, on="IAId", how="left"))
assert df[["Gender", "Banking Relationship", "Investment Advisor"]].notna().all().all()
df[["Client ID", "Gender", "Banking Relationship", "Investment Advisor"]].head()

,Client ID,Gender,Banking Relationship,Investment Advisor
0,IND81288,Male,Retail,Victor Dean
1,IND65833,Male,Institutional,Jeremy Porter
2,IND47499,Female,Private Bank,Ernest Knight
3,IND72498,Male,Commercial,Eric Shaw
4,IND60181,Female,Retail,Kevin Kim


## 4. Feature engineering

| Feature | Definition | Why |
|---|---|---|
| `total_loan` | Bank Loans + Business Lending + Credit Card Balance | Total credit exposure per client |
| `total_deposit` | Bank Deposits + Saving + Checking + Foreign Currency | Total funds held |
| `net_position` | total_deposit − total_loan | Is the client a net depositor or a net borrower? |
| `loan_to_deposit` | total_loan / total_deposit | Funding cover per client |
| `loan_to_income` | Bank Loans / Estimated Income | Affordability: how many years of income the loan represents |
| `income_band` | Low (< 100K), Mid (100K–300K), High (≥ 300K) | Segmentation |
| `age_group` | 18–30, 31–45, 46–60, 60+ | Segmentation |
| `tenure_years` | years from Joined Bank to 31-Dec-2021 | Engagement / relationship length |
| `processing_fee_rate` | High 5%, Mid 3%, Low 1% (from Fee Structure) | Fee revenue estimate |
| `est_fee_revenue` | total_loan × processing_fee_rate | Revenue contribution |
| `relationship_segment` | 2×2 of loans vs deposits (median split) | Product-mix segment |
| `underrated_risk_flag` | loan_to_income > 6 **and** Risk Weighting ≤ 2 | Clients the bank rates low-risk but who are highly leveraged |

In [12]:
SNAPSHOT = pd.Timestamp("2021-12-31")

df["total_loan"] = df["Bank Loans"] + df["Business Lending"] + df["Credit Card Balance"]
df["total_deposit"] = (df["Bank Deposits"] + df["Saving Accounts"]
                       + df["Checking Accounts"] + df["Foreign Currency Account"])
df["net_position"] = df["total_deposit"] - df["total_loan"]
# guard against divide-by-zero: clients with zero deposits get NaN, not infinity
df["loan_to_deposit"] = df["total_loan"] / df["total_deposit"].replace(0, np.nan)
df["loan_to_income"] = df["Bank Loans"] / df["Estimated Income"]

df["income_band"] = pd.cut(df["Estimated Income"], [0, 100_000, 300_000, np.inf],
                           right=False, labels=["Low", "Mid", "High"])
df["age_group"] = pd.cut(df["Age"], [0, 30, 45, 60, 200], labels=["18-30", "31-45", "46-60", "60+"])
df["tenure_years"] = ((SNAPSHOT - df["Joined Bank"]).dt.days / 365.25).round(1)
df["tenure_band"] = pd.cut(df["tenure_years"], [-1, 5, 10, 20, 100],
                           labels=["0-5 yrs", "5-10 yrs", "10-20 yrs", "20+ yrs"])
df["join_year"] = df["Joined Bank"].dt.year

df["processing_fee_rate"] = df["Fee Structure"].map({"High": 0.05, "Mid": 0.03, "Low": 0.01})
df["est_fee_revenue"] = df["total_loan"] * df["processing_fee_rate"]

loan_med, dep_med = df["total_loan"].median(), df["total_deposit"].median()
df["relationship_segment"] = np.select(
    [(df.total_loan >= loan_med) & (df.total_deposit >= dep_med),
     (df.total_loan >= loan_med) & (df.total_deposit < dep_med),
     (df.total_loan < loan_med) & (df.total_deposit >= dep_med)],
    ["Full relationship", "Borrower-heavy", "Saver"], default="Low engagement")

df["underrated_risk_flag"] = (df["loan_to_income"] > 6) & (df["Risk Weighting"] <= 2)
df[["total_loan", "total_deposit", "loan_to_income", "income_band", "age_group",
    "tenure_years", "relationship_segment", "underrated_risk_flag"]].head()

,total_loan,total_deposit,loan_to_income,income_band,age_group,tenure_years,relationship_segment,underrated_risk_flag
0,"1,911,202.76","2,709,028.94",10.30,Low,18-30,2.70,Full relationship,True
1,"3,273,398.41","1,276,801.63",4.38,Mid,18-30,20.10,Full relationship,False
2,"1,605,422.16","1,968,202.41",6.19,Mid,18-30,11.90,Full relationship,False
3,"1,273,802.35","2,388,513.65",0.34,High,31-45,2.80,Full relationship,False
4,"2,726,493.56","1,092,790.37",8.02,Mid,46-60,9.40,Full relationship,False


In [13]:
for c in ["income_band", "age_group", "tenure_band", "relationship_segment", "underrated_risk_flag"]:
    print(df[c].value_counts().sort_index().to_dict())

{'Low': 1027, 'Mid': 1517, 'High': 456}
{'18-30': 610, '31-45': 640, '46-60': 666, '60+': 1084}
{'0-5 yrs': 850, '5-10 yrs': 575, '10-20 yrs': 1069, '20+ yrs': 506}
{'Borrower-heavy': 491, 'Full relationship': 1009, 'Low engagement': 1009, 'Saver': 491}
{False: 2405, True: 595}


## 5. Export the clean table

In [14]:
out = df.rename(columns=lambda c: c.strip().lower().replace(" ", "_"))
out = out.drop(columns=["genderid", "brid", "iaid"])
out.to_csv("../data/processed/banking_clean.csv", index=False)
print(out.shape)
out.columns.tolist()

(3000, 41)


['client_key',
 'client_id',
 'name',
 'age',
 'location_id',
 'joined_bank',
 'banking_contact',
 'nationality',
 'occupation',
 'fee_structure',
 'loyalty_classification',
 'estimated_income',
 'superannuation_savings',
 'amount_of_credit_cards',
 'credit_card_balance',
 'bank_loans',
 'bank_deposits',
 'checking_accounts',
 'saving_accounts',
 'foreign_currency_account',
 'business_lending',
 'properties_owned',
 'risk_weighting',
 'client_id_reused',
 'gender',
 'banking_relationship',
 'investment_advisor',
 'total_loan',
 'total_deposit',
 'net_position',
 'loan_to_deposit',
 'loan_to_income',
 'income_band',
 'age_group',
 'tenure_years',
 'tenure_band',
 'join_year',
 'processing_fee_rate',
 'est_fee_revenue',
 'relationship_segment',
 'underrated_risk_flag']

## Summary
- 3,000 records, no missing values, no fully duplicated rows, all lookup keys valid.
- **Key issue found:** 59 `Client ID`s are reused by 2 different people (119 rows). Kept all rows and added a surrogate key; a distinct-count of Client ID would undercount clients by 60.
- Dates parsed; tenure computed against a fixed snapshot (31-Dec-2021) for reproducibility.
- 14 engineered features, including affordability (`loan_to_income`) and an **under-rated risk** flag used in the risk analysis.
- Output: `data/processed/banking_clean.csv` → used by `02_eda_segmentation_risk.ipynb`, the PostgreSQL scripts and Tableau.